# 🗄️ Módulo 2: Bases de Datos Relacionales (SQL con SQLite)
**Objetivo:** Darle persistencia a los datos de la clase `Producto` para que no se borren al apagar el PC.

### ¿Qué es SQLite?
Es un motor de base de datos SQL completo que guarda toda la información en un solo archivo local (`.db`). 

*Nota: Para ver la base de datos visualmente en VS Code, instalen la extensión "SQLite Viewer".*

### El Estándar CRUD:
1. **C**reate (Crear) -> `INSERT INTO`
2. **R**ead (Leer) -> `SELECT`
3. **U**pdate (Actualizar) -> `UPDATE`
4. **D**elete (Eliminar) -> `DELETE`

In [3]:
# ==============================================================================
# CRUD EN SQLITE (Módulo Producto)
# ==============================================================================

# Primero debemos importar la librería para manejar SQLite en Python
import sqlite3

# 1. CONEXIÓN Y CREACIÓN DE LA TABLA

# Conectamos a la BD (si el archivo no existe, Python lo crea automáticamente)

# Usamos sqlite3.connect para establecer la conexión con la base de datos "cafeteria_sabana.db"
conexion = sqlite3.connect("cafeteria_sabana.db")

# Usamos conexion.cursor() para crear un cursor, que es el objeto que nos permite ejecutar comandos SQL en la base de datos.
cursor = conexion.cursor()

## Creamos la tabla 'productos' usando lenguaje SQL

# Usamos cursor.execute() para ejecutar un comando SQL que crea la tabla 'productos' si no existe ya. 
# La tabla tiene las siguientes columnas:
# - id_producto: un entero que se autoincrementa y es la clave primaria.
# - nombre: un texto que no puede ser nulo.
# - precio: un número real que no puede ser nulo.
# - stock: un entero que no puede ser nulo.

cursor.execute('''
    CREATE TABLE IF NOT EXISTS productos (
        id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        precio REAL NOT NULL,
        stock INTEGER NOT NULL
    )
''')

# Observen el uso de triple comillas (''') para escribir la consulta SQL en varias líneas, lo que mejora la legibilidad del código.

# Usamos conexion.commit() para guardar los cambios realizados en la base de datos. 
# Esto es importante después de ejecutar comandos que modifican la estructura o los datos de la base de datos.
conexion.commit()

print("✅ Tabla 'productos' lista en la base de datos.")

# 2. CREATE (Insertar datos)

# Usamos '?' para evitar ataques de Inyección SQL (Seguridad). 
# Es decir, en lugar de concatenar directamente los valores en la consulta SQL, 
# usamos marcadores de posición ('?') 
# y luego pasamos los valores como una tupla en el segundo argumento de cursor.execute(). 
# Esto asegura que los valores se manejen de manera segura 
# y evita que un atacante pueda inyectar código malicioso a través de los datos de entrada.

def insertar_producto(nombre, precio, stock):

    cursor.execute('''
        INSERT INTO productos (nombre, precio, stock) 
        VALUES (?, ?, ?)
    ''', (nombre, precio, stock))

    conexion.commit()
    print(f"➕ Producto '{nombre}' insertado correctamente.")


# 3. READ (Leer datos)

# SELECT * FROM productos es la consulta SQL que se utiliza para seleccionar todos los registros de la tabla 'productos'. 
# El asterisco (*) es un comodín que indica que queremos seleccionar todas las columnas de la tabla. 
# Al ejecutar esta consulta, se obtendrá un conjunto de resultados que contiene todas las filas y columnas de la tabla 'productos'.

def leer_productos():

    cursor.execute('SELECT * FROM productos')

# Usamos cursor.fetchall() para obtener todos los resultados de la consulta SQL ejecutada. 
# Esto devuelve una lista de tuplas, donde cada tupla representa una fila de la tabla 'productos'.

    resultados = cursor.fetchall()

    print("\n📦 INVENTARIO ACTUAL EN BASE DE DATOS:")

    for fila in resultados:

        print(f"ID: {fila[0]} | Nombre: {fila[1]} | Precio: ${fila[2]:,.2f} | Stock: {fila[3]}")

    print("-" * 40)


# 4. UPDATE (Actualizar datos)

# UPDATE se usa para modificar los datos existentes en una tabla. 
# En este caso, queremos actualizar el precio de un producto específico identificado por su id_producto.

# SET precio = ? se usa para indicar que queremos cambiar el valor de la columna 'precio' a un nuevo valor que se pasará como argumento.

# WHERE id_producto = ? se usa para especificar qué fila(s) queremos actualizar. 
# En este caso, solo se actualizará el producto cuyo id_producto coincida con el valor que se pasará como argumento.

def actualizar_precio(id_producto, nuevo_precio):

    cursor.execute('''
        UPDATE productos 
        SET precio = ? 
        WHERE id_producto = ?
    ''', (nuevo_precio, id_producto))

    conexion.commit()
    
    print(f"🔄 Precio actualizado para el producto ID {id_producto}.")


# 5. DELETE (Eliminar datos)

# Se usa DELETE FROM para eliminar registros de una tabla. 
# En este caso, queremos eliminar un producto específico identificado por su id_producto.

def eliminar_producto(id_producto):

    cursor.execute('''
        DELETE FROM productos 
        WHERE id_producto = ?
    ''', (id_producto,))

    conexion.commit()

    print(f"❌ Producto ID {id_producto} eliminado de la base de datos.")



✅ Tabla 'productos' lista en la base de datos.


In [4]:
# ==============================================================================
# PRUEBA DEL CRUD 
# ==============================================================================

# Limpiamos la tabla para la prueba (Opcional, solo para este ejercicio)

cursor.execute('DELETE FROM productos') 

# Ejecutamos CREATE
insertar_producto("Café Tostao", 5000, 50)
insertar_producto("Chocolatina Jet", 1200, 100)

# Ejecutamos READ
leer_productos()

# Ejecutamos UPDATE (Asumiendo que Café Tostao es el ID 1)
actualizar_precio(1, 5500)
leer_productos()

# Ejecutamos DELETE (Eliminamos la Chocolatina, ID 2)
eliminar_producto(2)
leer_productos()

# Cerramos la conexión por seguridad
conexion.close()

➕ Producto 'Café Tostao' insertado correctamente.
➕ Producto 'Chocolatina Jet' insertado correctamente.

📦 INVENTARIO ACTUAL EN BASE DE DATOS:
ID: 3 | Nombre: Café Tostao | Precio: $5,000.00 | Stock: 50
ID: 4 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
----------------------------------------
🔄 Precio actualizado para el producto ID 1.

📦 INVENTARIO ACTUAL EN BASE DE DATOS:
ID: 3 | Nombre: Café Tostao | Precio: $5,000.00 | Stock: 50
ID: 4 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
----------------------------------------
❌ Producto ID 2 eliminado de la base de datos.

📦 INVENTARIO ACTUAL EN BASE DE DATOS:
ID: 3 | Nombre: Café Tostao | Precio: $5,000.00 | Stock: 50
ID: 4 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
----------------------------------------


### 📝 TAREA 1: Autogestión SQL

Tomando como base el código anterior, deben crear las tablas y hacer las operaciones CRUD (Create, Read, Update, Delete) en SQLite para los módulos de:

1. `Cliente` (id_cliente, nombre, tipo_cliente)

2. `Proveedor` (nit, nombre_empresa, ciudad)

3. `CarritoDeCompras` (id_venta, id_cliente, total_pagado) -> *Pista: Aquí usarán Llaves Foráneas (Foreign Keys).*
